# 04 — Gold Layer: fact_application_pipeline (Accumulating Snapshot)

**Purpose:** Build and maintain the accumulating snapshot fact table tracking
the full student application pipeline from lead to enrolment.

**Kimball fact type:** Accumulating Snapshot
- One row per student application — inserted at lead capture
- Row is UPDATED as each milestone completes via Delta MERGE
- Multiple date FKs — one per milestone (unresolved = 0, TBD row in dim_date)
- Lag facts stored as integers at ETL time — never calculated in DAX

**Milestones tracked:**
1. Lead captured (HubSpot)
2. Campus visit (HubSpot)
3. Application submitted (Dynamics)
4. Offer made (Dynamics)
5. Decision recorded — accepted or rejected (Dynamics)
6. Enrolled or withdrawn (Paradigm)

**Why MERGE and not overwrite?**
The accumulating snapshot pattern requires updating specific columns
on an existing row as milestones fire — leaving all other columns untouched.
Full reload would lose the history of which milestone arrived when.

## Parameters

In [ ]:
pipeline_run_date = "2024-01-15"

## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

run_date = pipeline_run_date
print(f"Pipeline run date: {run_date}")

FACT_PIPELINE = "gold.fact_application_pipeline"


def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name)
        return True
    except Exception:
        return False


def date_to_key(date_col):
    """
    Convert a DATE column to YYYYMMDD integer key for dim_date lookup.
    NULL dates become 0 (TBD row in dim_date — FK must never be NULL).
    """
    return F.when(
        date_col.isNotNull(),
        F.date_format(date_col, "yyyyMMdd").cast(IntegerType())
    ).otherwise(F.lit(0))


def lag_days(end_col, start_col):
    """
    Calculate lag in days between two date columns.
    Returns NULL if either date is not yet known (still TBD).
    Once calculated, this value is stored permanently — never recalculated.
    """
    return F.when(
        end_col.isNotNull() & start_col.isNotNull(),
        F.datediff(end_col, start_col)
    ).otherwise(F.lit(None).cast(IntegerType()))


print("Setup complete.")

## Step 1 — Build Incoming Pipeline Snapshot

Join Silver tables to construct the full pipeline state for each student.
Then resolve dimension surrogate keys.

This represents the current known state of every application.
The MERGE will compare this against what is already in the fact table
and update only the columns that have changed.

In [ ]:
# Load Silver tables
silver_leads  = spark.table("silver.student").filter("_is_current = true")
silver_apps   = spark.table("silver.application")
silver_enrol  = spark.table("silver.enrolment")

# Load dimension lookups
dim_student     = spark.table("gold.dim_student").filter("_is_current = true").select("student_sk", "student_id")
dim_course      = spark.table("gold.dim_course").select("course_sk", "course_id")
dim_campus      = spark.table("gold.dim_campus").select("campus_sk", "campus_id")
dim_intake      = spark.table("gold.dim_intake").select("intake_sk", "intake_id")
dim_lead_source = spark.table("gold.dim_lead_source").select("lead_source_sk", "source_name")

# Build one row per lead (all 10 students)
# Left join applications and enrolments — students without these will have NULLs
pipeline_raw = (
    silver_leads
    .select(
        "student_id",
        "student_email",
        "lead_source",
        F.col("hubspot_lead_id").alias("lead_id"),
    )
    # Join application data — one application per student in our dataset
    .join(
        silver_apps.select(
            "student_id",
            "application_id",
            "course_code",
            "campus_code",
            "intake_code",
            "application_date",
            "offer_date",
            "decision_date",
            "decision_outcome",
        ),
        on="student_id",
        how="left"
    )
    # Join enrolment data
    .join(
        silver_enrol.select(
            "student_id",
            "enrolment_date",
            "enrolment_status",
            "withdrawal_date",
        ),
        on="student_id",
        how="left"
    )
)

# Resolve lead_date and visit_date from HubSpot bronze directly
# (these columns live in HubSpot, not in silver.student)
hs_dates = (
    spark.table("bronze.hubspot_leads")
    .filter(F.col("_ingestion_date") == run_date)
    .select(
        F.col("lead_id"),
        F.col("lead_date").cast("date"),
        F.col("visit_date").cast("date"),
    )
)

pipeline_raw = pipeline_raw.join(hs_dates, on="lead_id", how="left")

print(f"Pipeline rows (one per student): {pipeline_raw.count()}")

In [ ]:
# Resolve dimension surrogate keys + calculate date keys and lag facts
pipeline_snapshot = (
    pipeline_raw
    # Dimension SK lookups
    .join(dim_student,     on="student_id",  how="left")
    .join(dim_course,      pipeline_raw.course_code == dim_course.course_id,  how="left")
    .join(dim_campus,      pipeline_raw.campus_code == dim_campus.campus_id,  how="left")
    .join(dim_intake,      pipeline_raw.intake_code == dim_intake.intake_id,  how="left")
    .join(dim_lead_source, pipeline_raw.lead_source == dim_lead_source.source_name, how="left")

    # Convert milestone dates to dim_date keys (NULL → 0 = TBD)
    .withColumn("lead_date_key",         date_to_key(F.col("lead_date")))
    .withColumn("visit_date_key",        date_to_key(F.col("visit_date")))
    .withColumn("application_date_key",  date_to_key(F.col("application_date")))
    .withColumn("offer_date_key",        date_to_key(F.col("offer_date")))
    .withColumn("decision_date_key",     date_to_key(F.col("decision_date")))
    .withColumn("enrolment_date_key",    date_to_key(F.col("enrolment_date")))

    # Calculate lag facts — stored permanently when both dates are known
    .withColumn("lead_to_visit_days",          lag_days(F.col("visit_date"),       F.col("lead_date")))
    .withColumn("lead_to_application_days",    lag_days(F.col("application_date"), F.col("lead_date")))
    .withColumn("application_to_offer_days",   lag_days(F.col("offer_date"),       F.col("application_date")))
    .withColumn("offer_to_decision_days",      lag_days(F.col("decision_date"),    F.col("offer_date")))
    .withColumn("decision_to_enrolment_days",  lag_days(F.col("enrolment_date"),   F.col("decision_date")))
    .withColumn("lead_to_enrolment_days",      lag_days(F.col("enrolment_date"),   F.col("lead_date")))

    # Milestone flags (additive: SUM = count of students who reached this stage)
    .withColumn("is_lead",               F.lit(1))
    .withColumn("is_visited",            F.when(F.col("visit_date").isNotNull(),        F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_applied",            F.when(F.col("application_date").isNotNull(),  F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_offered",            F.when(F.col("offer_date").isNotNull(),        F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_accepted",           F.when(F.col("decision_outcome") == "Accepted", F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_enrolled",           F.when(F.col("enrolment_date").isNotNull(),    F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_withdrawn_pre_enrol",
        F.when(
            (F.col("decision_outcome") != "Accepted") & F.col("decision_outcome").isNotNull(),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn("_loaded_at",      F.current_timestamp())
    .withColumn("_last_updated_at", F.current_timestamp())

    # Use lead_id as the pipeline natural key for students without an application
    # Use application_id for students who have applied
    .withColumn("pipeline_natural_key",
        F.coalesce(F.col("application_id"), F.col("lead_id"))
    )

    .select(
        "pipeline_natural_key",
        "student_sk",
        F.coalesce(F.col("course_sk"),      F.lit(-1)).alias("course_sk"),
        F.coalesce(F.col("campus_sk"),      F.lit(-1)).alias("campus_sk"),
        F.coalesce(F.col("intake_sk"),      F.lit(-1)).alias("intake_sk"),
        F.coalesce(F.col("lead_source_sk"), F.lit(-1)).alias("lead_source_sk"),
        "lead_date_key", "visit_date_key", "application_date_key",
        "offer_date_key", "decision_date_key", "enrolment_date_key",
        F.col("lead_id"),
        F.col("application_id"),
        "lead_to_visit_days", "lead_to_application_days",
        "application_to_offer_days", "offer_to_decision_days",
        "decision_to_enrolment_days", "lead_to_enrolment_days",
        "is_lead", "is_visited", "is_applied", "is_offered",
        "is_accepted", "is_enrolled", "is_withdrawn_pre_enrol",
        "_loaded_at", "_last_updated_at",
    )
)

print(f"Pipeline snapshot rows: {pipeline_snapshot.count()}")
pipeline_snapshot.select(
    "pipeline_natural_key", "lead_date_key", "application_date_key",
    "offer_date_key", "enrolment_date_key",
    "lead_to_enrolment_days",
    "is_lead", "is_applied", "is_offered", "is_enrolled"
).show(15, truncate=False)

## Step 2 — MERGE into fact_application_pipeline

**MERGE logic:**
- `WHEN MATCHED` — update milestone columns that have progressed (date_key moved from 0 to a real date)
  Lag facts are only written when both endpoint dates are now known — they are never overwritten once set.
- `WHEN NOT MATCHED` — insert new row with lead milestone populated, all others defaulted to 0 or NULL

This is what makes it an accumulating snapshot — the same row is updated across multiple pipeline runs.

In [ ]:
if not table_exists(FACT_PIPELINE):
    # ── First run: assign surrogate keys and insert all rows ───────────────
    print("[fact_pipeline] First run — creating table")

    w = Window.orderBy("pipeline_natural_key")
    first_load = pipeline_snapshot.withColumn(
        "pipeline_sk", F.row_number().over(w).cast(IntegerType())
    )

    first_load.write.format("delta").mode("overwrite").saveAsTable(FACT_PIPELINE)
    print(f"[fact_pipeline] Inserted {first_load.count()} rows")

else:
    # ── Subsequent runs: MERGE ─────────────────────────────────────────────
    print("[fact_pipeline] Existing table found — running accumulating snapshot MERGE")

    target = DeltaTable.forName(spark, FACT_PIPELINE)

    (
        target.alias("t").merge(
            pipeline_snapshot.alias("s"),
            "t.pipeline_natural_key = s.pipeline_natural_key"
        )
        .whenMatchedUpdate(set={
            # Visit milestone — only update if newly resolved
            "visit_date_key": """
                CASE WHEN s.visit_date_key != 0 AND t.visit_date_key = 0
                     THEN s.visit_date_key ELSE t.visit_date_key END""",
            "lead_to_visit_days": """
                CASE WHEN s.visit_date_key != 0 AND t.visit_date_key = 0
                     THEN s.lead_to_visit_days ELSE t.lead_to_visit_days END""",
            "is_visited": """
                CASE WHEN s.visit_date_key != 0 THEN 1 ELSE t.is_visited END""",

            # Application milestone
            "application_date_key": """
                CASE WHEN s.application_date_key != 0 AND t.application_date_key = 0
                     THEN s.application_date_key ELSE t.application_date_key END""",
            "application_id": """
                CASE WHEN s.application_id IS NOT NULL AND t.application_id IS NULL
                     THEN s.application_id ELSE t.application_id END""",
            "lead_to_application_days": """
                CASE WHEN s.application_date_key != 0 AND t.application_date_key = 0
                     THEN s.lead_to_application_days ELSE t.lead_to_application_days END""",
            "is_applied": """
                CASE WHEN s.application_date_key != 0 THEN 1 ELSE t.is_applied END""",

            # Offer milestone
            "offer_date_key": """
                CASE WHEN s.offer_date_key != 0 AND t.offer_date_key = 0
                     THEN s.offer_date_key ELSE t.offer_date_key END""",
            "application_to_offer_days": """
                CASE WHEN s.offer_date_key != 0 AND t.offer_date_key = 0
                     THEN s.application_to_offer_days ELSE t.application_to_offer_days END""",
            "is_offered": """
                CASE WHEN s.offer_date_key != 0 THEN 1 ELSE t.is_offered END""",

            # Decision milestone
            "decision_date_key": """
                CASE WHEN s.decision_date_key != 0 AND t.decision_date_key = 0
                     THEN s.decision_date_key ELSE t.decision_date_key END""",
            "offer_to_decision_days": """
                CASE WHEN s.decision_date_key != 0 AND t.decision_date_key = 0
                     THEN s.offer_to_decision_days ELSE t.offer_to_decision_days END""",
            "is_accepted": """
                CASE WHEN s.is_accepted = 1 THEN 1 ELSE t.is_accepted END""",
            "is_withdrawn_pre_enrol": """
                CASE WHEN s.is_withdrawn_pre_enrol = 1 THEN 1 ELSE t.is_withdrawn_pre_enrol END""",

            # Enrolment milestone
            "enrolment_date_key": """
                CASE WHEN s.enrolment_date_key != 0 AND t.enrolment_date_key = 0
                     THEN s.enrolment_date_key ELSE t.enrolment_date_key END""",
            "decision_to_enrolment_days": """
                CASE WHEN s.enrolment_date_key != 0 AND t.enrolment_date_key = 0
                     THEN s.decision_to_enrolment_days ELSE t.decision_to_enrolment_days END""",
            "lead_to_enrolment_days": """
                CASE WHEN s.enrolment_date_key != 0 AND t.enrolment_date_key = 0
                     THEN s.lead_to_enrolment_days ELSE t.lead_to_enrolment_days END""",
            "is_enrolled": """
                CASE WHEN s.enrolment_date_key != 0 THEN 1 ELSE t.is_enrolled END""",

            # Always update audit timestamp
            "_last_updated_at": "current_timestamp()",
        })
        .whenNotMatchedInsert(values={
            "pipeline_sk":             f"(SELECT COALESCE(MAX(pipeline_sk), 0) + 1 FROM {FACT_PIPELINE})",
            "pipeline_natural_key":    "s.pipeline_natural_key",
            "student_sk":              "s.student_sk",
            "course_sk":               "s.course_sk",
            "campus_sk":               "s.campus_sk",
            "intake_sk":               "s.intake_sk",
            "lead_source_sk":          "s.lead_source_sk",
            "lead_date_key":           "s.lead_date_key",
            "visit_date_key":          "s.visit_date_key",
            "application_date_key":    "s.application_date_key",
            "offer_date_key":          "s.offer_date_key",
            "decision_date_key":       "s.decision_date_key",
            "enrolment_date_key":      "s.enrolment_date_key",
            "lead_id":                 "s.lead_id",
            "application_id":          "s.application_id",
            "lead_to_visit_days":          "s.lead_to_visit_days",
            "lead_to_application_days":    "s.lead_to_application_days",
            "application_to_offer_days":   "s.application_to_offer_days",
            "offer_to_decision_days":      "s.offer_to_decision_days",
            "decision_to_enrolment_days":  "s.decision_to_enrolment_days",
            "lead_to_enrolment_days":      "s.lead_to_enrolment_days",
            "is_lead":                 "s.is_lead",
            "is_visited":              "s.is_visited",
            "is_applied":              "s.is_applied",
            "is_offered":              "s.is_offered",
            "is_accepted":             "s.is_accepted",
            "is_enrolled":             "s.is_enrolled",
            "is_withdrawn_pre_enrol":  "s.is_withdrawn_pre_enrol",
            "_loaded_at":              "current_timestamp()",
            "_last_updated_at":        "current_timestamp()",
        })
        .execute()
    )
    print("[fact_pipeline] MERGE complete")

## Verification

In [ ]:
fact = spark.table(FACT_PIPELINE)
print(f"Total rows: {fact.count()} (should be 10 — one per student)")
print()

# Funnel summary
print("Pipeline funnel:")
fact.agg(
    F.sum("is_lead").alias("total_leads"),
    F.sum("is_visited").alias("visited"),
    F.sum("is_applied").alias("applied"),
    F.sum("is_offered").alias("offered"),
    F.sum("is_accepted").alias("accepted"),
    F.sum("is_enrolled").alias("enrolled"),
).show(truncate=False)

# Lag facts
print("Lag facts (NULL = milestone not yet reached):")
fact.select(
    "pipeline_natural_key",
    "lead_to_application_days",
    "application_to_offer_days",
    "offer_to_decision_days",
    "decision_to_enrolment_days",
    "lead_to_enrolment_days"
).orderBy("pipeline_natural_key").show(15, truncate=False)

## Summary

`fact_application_pipeline` built and verified.
One row per student. Milestone flags and lag facts correctly populated.

**To test the accumulating snapshot pattern:**
1. Update a student's CSV to add a new milestone (e.g. give STU003 an offer date)
2. Re-run the full pipeline with a new `pipeline_run_date`
3. The MERGE will update only that student's offer columns — all others unchanged

**Next steps:** Run `05_gold_fact_enrolment.ipynb` and `06_gold_fact_attendance.ipynb`.